#### Subjects
 - 50000R
 - 50017L
 - 50034R
#### Parameters
 - 1.5 layars of T10's
 - d0 = 5
#### Poses
 - Flexion
 - Extension
 - abduction
 - adduction
 - pinch load
#### End criteria
 - 0.8 mm
 - 50 N

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

from phd_helpers.AbaqusPostprocessing import (
    inp2pv, get_field_path, get_field_df, add_field_to_mesh, parse_wallclock_time, parse_final_step_time, get_history_path
)
from phd_helpers.AbaqusPreprocessing import parse_memory_estimate

In [17]:
DIR = Path('../../../../Computational/InpPipeline/outputs/initialFEAstuff/poses_d5CAsubs')

bones = ['tpm']
subjects = ['50000R', '50017L', '50034R']
ids = [
    ('d5', '0'), # (run_id_mesh, run_id)
    ]
poses = ['flexion', 'extension', 'abduction', 'adduction', 'pinch_load']
step = 0
frame = -1

field_metrics = ["CPRESS", "U"]


data = []
for subject in subjects:
    for bone in bones:
        instance = f"{bone.upper()}_INST"
        for pose in poses:
            for run_id_mesh, run_id in ids:
                try:
                    job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

                    path_inp = DIR / f'inpFiles/{subject}/inp'
                    path_job = path_inp / job_name
                    path_csv = path_job / 'resultCSVs'
                    input_path = path_job / job_name.with_suffix('.inp')
                    dat_path = path_job / job_name.with_suffix('.dat')
                    sta_path = path_job / job_name.with_suffix('.sta')

                    # RESULTS #
                    meshes = inp2pv(input_path)
                    mesh = meshes[bone]

                    # Field data
                    for metric in field_metrics:
                        field_path = get_field_path(path_csv, metric, step, frame, instance)
                        field_df = get_field_df(field_path)
                        add_field_to_mesh(mesh, field_df)
                    mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)

                    # History data
                    history_data = pd.read_csv(get_history_path(path_csv, step))
                    RF_data = history_data[history_data['historyOutputKey']=='RF1']
                    CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']


                    data.append({
                        'subject': subject,
                        'job': job_name,
                        'P': mesh['CPRESS'].max(),
                        'A': CAREA_data['value'].iloc[frame],
                        'F' : np.abs(RF_data['value'].iloc[frame]),
                        'd' : parse_final_step_time(sta_path),
                        'nodes' : mesh.n_points,
                        'elements' : mesh.n_cells,
                        'memory' : parse_memory_estimate(dat_path)['memory_to_minimize_io_gb'],
                        'runtime' : parse_wallclock_time(dat_path),
                    })
                except: pass

df = pd.DataFrame(data)

In [18]:
df

,subject,job,P,A,F,d,nodes,elements,memory,runtime
0,50000R,d5-flexion-0,2.825066,56.927898,49.970608,0.140,84193,57027,9.727,2153.0
1,50000R,d5-extension-0,1.978922,42.436710,34.708260,0.203,84193,57027,9.725,5544.0
2,50000R,d5-abduction-0,3.535621,50.648445,49.999218,0.192,84193,57027,9.726,4110.0
3,50000R,d5-adduction-0,3.134415,52.926769,47.974667,0.597,84193,57027,9.723,8413.0
4,50000R,d5-pinch_load-0,3.832304,37.547939,34.208389,0.283,84193,57027,9.725,7162.0
5,50017L,d5-extension-0,2.138328,51.298264,36.154491,0.265,100137,68575,11.789,7603.0
6,50017L,d5-abduction-0,2.856802,35.041702,31.751244,0.358,100137,68575,11.786,10207.0
7,50017L,d5-adduction-0,3.035224,48.467674,38.049072,0.330,100137,68575,11.789,5615.0


# Plan:
tweak params for extension (50000R) - find robustest ones
### Study 1
 - Loop over:
    - max step
    - extrapolation
    - convert_sdi
    - friction
    - hybrid modified

    - stabilisation ?
    - Contact normal behavior ?
### Study 2
 - Run best (30) params but with C3D10M bone elements with and without stabilisation
 - Were also unintentionally compressible D1=0.02
### Study 3
 - higher stabilise factors = [5e-3, 5e-2]
 - D1 = [0, 0.02]
 - friction= [0.1, 0.0]
### Study 4
 - pressure-overclosure c0=0.002 mm, p0=0.02 MPa
 - SLIDING = FINITE, SMALL - can't use small
 - EXPONENTIAL, HARD
 - H, MH
 - Stabilise = False
### Study 5
 - C3D4H 3.5 n_tets (51000 nodes)
### Study 6
 - C3D10M/H thickness_or_max (87000 nodes)
 - C3D10M/H 2 n_tets


In [5]:
#mesh_path = '../../../../Computational/MeshPipeline/outputs/ParamOptimisation/optimise_d0/d5best'
import subprocess
inpMain_path = '../../../../Computational/InpPipeline/main.py'
subprocess.run(['python', inpMain_path])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/inpPipeline/set_parameters/parameters.json
Full parameter file saved to outputs/tweak_50000R_ext/study6-inp/params/full_params.json

SUBJECT: 50000R
	MESH: 2T
		RUN ID: 00
			Runtime: 15.171s - ok
		RUN ID: 01
			Runtime: 14.770s - ok
		RUN ID: 02
			Runtime: 14.944s - ok
		RUN ID: 03
			Runtime: 15.008s - ok
	MESH: 024M
		RUN ID: 00
			Runtime: 9.664s - ok
		RUN ID: 01
			Runtime: 9.671s - ok
		RUN ID: 02
			Runtime: 9.646s - ok
		RUN ID: 03
			Runtime: 9.731s - ok


CompletedProcess(args=['python', '../../../../Computational/InpPipeline/main.py'], returncode=0)

In [4]:
DIR = Path('outputs/tweak_50000R_ext')
studies = [1, 2, 3, 4, 5, 6]

bones = ['tpm']
subjects = ['50000R']#, '50017L', '50034R']

poses = ['extension']
step = 0
frame = -1

field_metrics = ["CPRESS", "U"]


data = []
for study in studies:
    for subject in subjects:
        for bone in bones:
            instance = f"{bone.upper()}_INST"
            for pose in poses:
                study_dir = DIR / f'study{study}-inp'
                prefix = '35T' if study == 5 else 'd5'
                ids = [ (x, y) for x, _, y in [j.with_suffix('').name.split('-') for j in study_dir.glob(f'**/inpFiles/{subject}/inp/**/*.inp')] ]
                
                for run_id_mesh, run_id in ids:
                    job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

                    path_inp = study_dir / f'inpFiles/{subject}/inp'
                    path_job = path_inp / job_name
                    path_csv = path_job / 'resultCSVs'
                    input_path = path_job / job_name.with_suffix('.inp')
                    dat_path = path_job / job_name.with_suffix('.dat')
                    sta_path = path_job / job_name.with_suffix('.sta')

                    # RESULTS #
                    meshes = inp2pv(input_path)
                    mesh = meshes[bone]

                    # Field data
                    for metric in field_metrics:
                        field_path = get_field_path(path_csv, metric, step, frame, instance)
                        field_df = get_field_df(field_path)
                        add_field_to_mesh(mesh, field_df)
                    mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)

                    # History data
                    history_data = pd.read_csv(get_history_path(path_csv, step))
                    RF_data = history_data[history_data['historyOutputKey']=='RF1']
                    CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']

                    with open(study_dir / f'params/loop_params/{run_id}.json', 'r') as f:
                        params = json.load(f)

                    data.append({
                        'study': study,
                        'subject': subject,
                        'run_id': run_id_mesh+'-'+run_id,
                        'P': mesh['CPRESS'].max(),
                        'A': CAREA_data['value'].iloc[frame],
                        'F' : np.abs(RF_data['value'].iloc[frame]),
                        'd' : parse_final_step_time(sta_path),
                        'nodes' : mesh.n_points,
                        'elements' : mesh.n_cells,
                        'memory' : parse_memory_estimate(dat_path)['memory_to_minimize_io_gb'],
                        'runtime' : parse_wallclock_time(dat_path),
                        'element_type': params['element_type']+params['cartilage_element_suffix'],
                        'friction': params['cartilage_friction'],
                        'D1': params['cartilage_material']['D1'],
                        'max_increment': params['max_increment'],
                        'convert_sdi': params['convert_sdi'],
                        'extrapolation': 'LINEAR' if params['extrapolation'] is None else params['extrapolation'],
                        'stabilise_factor': params['stabilize_factor'] if params.get('stabilize', False) else np.nan,
                        'contact_type': params.get( "contact_type", 'general'),
                        'overclosure': params['normal_data'] if params.get('overclosure', None) == 'EXPONENTIAL' else params.get('overclosure', np.nan),
                    })

df = pd.DataFrame(data)

In [6]:
df.sort_values('F')

,study,subject,run_id,P,A,F,d,nodes,elements,memory,runtime,element_type,friction,D1,max_increment,convert_sdi,extrapolation,stabilise_factor,contact_type,overclosure
52,6,50000R,2T-01,1.658993,40.374107,27.764565,0.177,140938,98314,21.096,22750.0,C3D10H,0.01,0.00,0.010,YES,LINEAR,NaN,general,HARD
12,1,50000R,d5-14,1.903232,41.501637,32.272739,0.188,84193,57027,9.725,4955.0,C3D10H,0.01,0.00,0.010,YES,LINEAR,NaN,general,NaN
13,1,50000R,d5-13,1.903232,41.501637,32.272739,0.188,84193,57027,9.725,5141.0,C3D10H,0.01,0.00,0.010,NO,NO,NaN,general,NaN
11,1,50000R,d5-15,1.903232,41.501637,32.272739,0.188,84193,57027,9.725,5092.0,C3D10H,0.01,0.00,0.010,YES,NO,NaN,general,NaN
10,1,50000R,d5-12,1.903232,41.501637,32.272739,0.188,84193,57027,9.725,5476.0,C3D10H,0.01,0.00,0.010,NO,LINEAR,NaN,general,NaN
49,6,50000R,2T-03,1.883218,41.664345,32.313934,0.186,140938,98314,21.609,18752.0,C3D10MH,0.01,0.00,0.010,YES,LINEAR,NaN,general,HARD
26,1,50000R,d5-10,1.924658,41.750446,32.755714,0.189,84193,57027,9.725,2512.0,C3D10H,0.01,0.00,0.025,YES,LINEAR,NaN,general,NaN
25,1,50000R,d5-11,1.924657,41.750450,32.755714,0.189,84193,57027,9.725,2594.0,C3D10H,0.01,0.00,0.025,YES,NO,NaN,general,NaN
4,1,50000R,d5-08,1.924658,41.750446,32.755714,0.189,84193,57027,9.725,2312.0,C3D10H,0.01,0.00,0.025,NO,LINEAR,NaN,general,NaN
3,1,50000R,d5-09,1.924657,41.750450,32.755714,0.189,84193,57027,9.725,2517.0,C3D10H,0.01,0.00,0.025,NO,NO,NaN,general,NaN


 - Modified elements are not compatible with regular elements if they share nodes
 - All the above study1 MH were done with regular C3D10 bone elements and C3D10MH cartilage elements
 - Running study2 with C3D10M bone elements and C3D10MH cartilage elements

- look at hourglass control methods in cae to see what inp looks like (24/10/2025 lecture) - C3D10I?
- lower max increment
- compressible
- selective reduced integration

## Load mesh and data

In [35]:
run_id = '01'
run_id_mesh = '35T'
study = 'study5'

DIR = Path(f'outputs/tweak_50000R_ext/{study}-inp')
bone = 'tpm'
subject = '50000R' #, '50017L', '50034R']
pose = 'extension'
step = 0
frame = -1
instance = f"{bone.upper()}_INST"
field_metrics = ["CPRESS", "U"]

job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

path_inp = DIR / f'inpFiles/{subject}/inp'
path_job = path_inp / job_name
path_csv = path_job / 'resultCSVs'
input_path = path_job / job_name.with_suffix('.inp')

# RESULTS #
meshes = inp2pv(input_path)
mesh = meshes[bone]

# Field data
for metric in field_metrics:
    field_path = get_field_path(path_csv, metric, step, frame, instance)
    field_df = get_field_df(field_path)
    add_field_to_mesh(mesh, field_df)
mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)
#mesh['VFmag'] = np.linalg.norm(np.column_stack((mesh['VF1'], mesh['VF2'], mesh['VF3'])), axis=1)

# History data
history_data = pd.read_csv(get_history_path(path_csv, step))
RF_data = history_data[history_data['historyOutputKey']=='RF1']
CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']
allie = history_data[history_data['historyOutputKey']=='ALLIE'] # internal energy
allsd = history_data[history_data['historyOutputKey']=='ALLSD'] # Static dissipation

### Surface wrinkling?
- MH elements cause surface wrinkling, H not so much

In [36]:
from phd_helpers.AbaqusPostprocessing import get_deformed_mesh
import pyvista as pv

print(RF_data['value'].min())
pl = pv.Plotter()
mesh_def = get_deformed_mesh(mesh)
pl.add_mesh(mesh_def, scalars='CPRESS', cmap='Blues', opacity=1)

#ab_id = mesh['abaqus_element_id'][62699]
#pl.add_points(mesh.points[mesh.cells.reshape(-1, 11)[:, 1:][bad_element]], color='cyan', point_size=10)
# mesh_def.points[mesh_def.cells.reshape(-1, 11)[:, 1:][mesh_def['tpm_CART_SURF']==1][2]]
pl.show()

-149.99853515625


Widget(value='<iframe src="http://localhost:56697/index.html?ui=P_0x35f389a00_8&reconnect=auto" class="pyvista…

In [131]:
from phd_helpers.paths import quadratic_to_linear_mesh, compute_edge_lengths

mesh_lin = quadratic_to_linear_mesh(mesh)
mesh_lin_shell = mesh_lin.extract_surface(algorithm=None)
cart_surf_lin = quadratic_to_linear_mesh(mesh_lin_shell.extract_cells(mesh_lin_shell['tpm_CART_SURF']==1))
Ls = compute_edge_lengths(cart_surf_lin)
print(f'Mean: {np.mean(Ls):.2f} mm')
print(f'Max: {np.max(Ls):.2f} mm')
print(f'Min: {np.min(Ls):.2f} mm')

cart_surf_lin.plot(color='gray')

Mean: 0.26 mm
Max: 0.49 mm
Min: 0.11 mm


Widget(value='<iframe src="http://localhost:54286/index.html?ui=P_0x36c826060_40&reconnect=auto" class="pyvist…

In [137]:
allsd['value'].values/allie['value'].values

/var/folders/md/chz9snx91svc73k94gsj2m7m0000gn/T/ipykernel_14244/2836946049.py:1: RuntimeWarning: invalid value encountered in divide
  allsd['value'].values/allie['value'].values


array([           nan, 6.47024634e+05, 2.88124728e+08, 4.65212593e+08,
       2.98589954e-01, 4.41070156e-03, 6.69316740e-04, 2.15381432e-04,
       9.52352382e-05, 5.86333375e-05, 9.87950986e-05, 4.73023917e-04,
       9.88842196e-04, 1.68998062e-03, 2.22130908e-03, 2.50545393e-03,
       2.57723460e-03, 2.50017668e-03, 2.35263430e-03, 2.18480204e-03,
       2.01823680e-03, 1.86244065e-03, 1.72049222e-03, 1.59227606e-03,
       1.47703703e-03, 1.45022444e-03, 1.41148484e-03, 1.35668190e-03,
       1.33698330e-03, 1.30850435e-03, 1.29808923e-03, 1.29808923e-03])

# poses_35Td5CA

In [13]:
DIR = Path('../../../../Computational/InpPipeline/outputs/initialFEAstuff/poses_35Td5CAsubs')

bones = ['tpm']
subjects = ['50000R', '50017L', '50034R']
ids = [
    ('35T', '00'), # (run_id_mesh, run_id)
    ]
poses = ['flexion', 'extension', 'abduction', 'adduction', 'pinch_load']
step = 0
frame = -1

field_metrics = ["CPRESS", "U"]


data = []
for subject in subjects:
    for bone in bones:
        instance = f"{bone.upper()}_INST"
        for pose in poses:
            for run_id_mesh, run_id in ids:
                job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

                path_inp = DIR / f'inpFiles/{subject}/inp'
                path_job = path_inp / job_name
                path_csv = path_job / 'resultCSVs'
                input_path = path_job / job_name.with_suffix('.inp')
                dat_path = path_job / job_name.with_suffix('.dat')
                sta_path = path_job / job_name.with_suffix('.sta')

                # RESULTS #
                meshes = inp2pv(input_path)
                mesh = meshes[bone]

                # Field data
                for metric in field_metrics:
                    field_path = get_field_path(path_csv, metric, step, frame, instance)
                    field_df = get_field_df(field_path)
                    add_field_to_mesh(mesh, field_df)
                mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)

                # History data
                history_data = pd.read_csv(get_history_path(path_csv, step))
                RF_data = history_data[history_data['historyOutputKey']=='RF1']
                CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']


                data.append({
                    'subject': subject,
                    'job': job_name,
                    'P': mesh['CPRESS'].max(),
                    'A': CAREA_data['value'].iloc[frame],
                    'F' : np.abs(RF_data['value'].iloc[frame]),
                    'd' : parse_final_step_time(sta_path),
                    'nodes' : mesh.n_points,
                    'elements' : mesh.n_cells,
                    'memory' : parse_memory_estimate(dat_path)['memory_to_minimize_io_gb'],
                    'runtime' : parse_wallclock_time(dat_path),
                })

df = pd.DataFrame(data)

In [ ]:
df # done with general contact, but maybe explicitly selecting primary / secondary surface is better?

,subject,job,P,A,F,d,nodes,elements,memory,runtime
0,50000R,35T-flexion-00,7.285730,68.972855,149.997940,0.237,51074,273301,6.367,3484.0
1,50000R,35T-extension-00,6.760465,60.922649,149.998535,0.346,51074,273301,6.365,4452.0
2,50000R,35T-abduction-00,8.547304,64.475677,149.996735,0.298,51074,273301,6.366,4268.0
3,50000R,35T-adduction-00,4.937740,57.397732,77.499878,0.630,51074,273301,6.361,5613.0
4,50000R,35T-pinch_load-00,10.360175,55.377514,135.261887,0.430,51074,273301,6.364,7044.0
5,50017L,35T-flexion-00,7.140409,51.027332,107.722916,0.436,60331,329970,7.721,6552.0
6,50017L,35T-extension-00,6.414884,71.166901,150.093079,0.393,60331,329970,7.724,5552.0
7,50017L,35T-abduction-00,7.420892,59.244247,124.513283,0.504,60331,329970,7.722,11943.0
8,50017L,35T-adduction-00,8.027984,67.686340,149.998566,0.444,60331,329970,7.724,6498.0
9,50017L,35T-pinch_load-00,6.982155,56.035744,100.113686,0.458,60331,329970,7.722,6018.0


In [ ]:
could run again with explicit
could also try equivalent amount of nodes but with thickness_or_max